# PyTorch GPU Basics

A five-minute tour: confirm your LabPod workspace can see its GPU, compare a CPU vs. GPU
tensor op, then train a tiny model for a few seconds so you can watch GPU utilization show
up on the LabPod dashboard while it runs.

Everything here uses synthetic random data - no dataset download required, so this works
even on an offline / air-gapped workspace.

## 1. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("No GPU visible - check the workspace's GPU mode in LabPod (None/Whole/Fractional/MIG).")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. CPU vs. GPU: a quick matrix multiply benchmark

Nothing scientific - just enough work to make the GPU line move on the LabPod resources page.

In [ ]:
import time


def bench(device, n=4096, iters=10):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(iters):
        c = a @ b
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) / iters


cpu_s = bench(torch.device("cpu"))
print(f"CPU:  {cpu_s * 1000:.1f} ms/iter")

if torch.cuda.is_available():
    gpu_s = bench(torch.device("cuda"))
    print(f"GPU:  {gpu_s * 1000:.1f} ms/iter")
    print(f"speedup: {cpu_s / gpu_s:.1f}x")

## 3. Train a tiny model for a few seconds

A small MLP on synthetic classification data. While this cell runs, check the workspace's
GPU utilization on the LabPod dashboard - it should show non-zero usage.

In [ ]:
import torch.nn as nn


n_samples, n_features, n_classes = 8192, 256, 10
X = torch.randn(n_samples, n_features, device=device)
y = torch.randint(0, n_classes, (n_samples,), device=device)

model = nn.Sequential(
    nn.Linear(n_features, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, n_classes),
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    opt.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    opt.step()
    if epoch % 40 == 0:
        acc = (logits.argmax(dim=1) == y).float().mean().item()
        print(f"epoch {epoch:3d}  loss {loss.item():.3f}  train_acc {acc:.2f}")

## Next steps

- Swap the synthetic data for your own dataset under `/work` (that's the persistent,
  host-backed mount - it survives stopping and starting the workspace).
- If you need extra Python packages, add them to `pytorch-scientific-ml/template/context/` as
  a `Dockerfile` + `requirements.txt` and rebuild the template, rather than `pip install`ing at
  runtime - anything installed outside `/work` or your persistent home is lost on the next
  workspace Start. See this repo's top-level README for why.
- Try another notebook in `pytorch-scientific-ml/notebooks/`, or another cookbook in this repo.